# PhysIR — DISTS on the SEED REPLICATES (Kaggle, one flow, resumable)

Closes the gap named in Ch.4 §4.7 / §4.9: **DISTS — the metric carrying §4.6.2's
metric-kinship argument — currently has no seed-robustness evidence at all.**

Adds exactly the **8 missing evals: 4 load-bearing arms × 2 seeds (1337, 2024)** at `@last`.
Seed 42 is **deliberately not recomputed** — it is already done by `dists_eval_kaggle.ipynb`
and would only burn session time.

| | seed 42 | seed 1337 | seed 2024 |
|---|---|---|---|
| `abl_baseline` | ✅ existing nb | ⬜ here | ⬜ here |
| `abl_full` | ✅ existing nb | ⬜ here | ⬜ here |
| `abl_percep` | ✅ existing nb | ⬜ here | ⬜ here |
| `abl_full_percep` | ✅ existing nb | ⬜ here | ⬜ here |

**The eval engine is lifted byte-identical from `dists_eval_kaggle.ipynb`** — same `[0,1]`
convention, same active `check_input_range` guard, same per-sample assertion, same
non-shuffled-loader hard guard. Both sets of numbers are therefore produced by the same code,
which is what makes them comparable across seeds.

**Inputs:** `physir-code`, `lolblur-dataset`, plus the Stage-6 outputs holding
`abl_*_s1337_latest.pt` / `abl_*_s2024_latest.pt` (Add Input → Your Work → the
`stage6_*` notebooks). **Accelerator: GPU T4 × 2.** ~15–30 min/arm → **~2–4 h**. Resumable.

**Output:** `per_image_dists_seeds/per_image_dists_{arm}_s{seed}.csv` (`idx, fname, dists`),
joinable on `idx` to the existing `per_image_abl_{arm}_s{seed}[_last].csv`.

## 1. Install

In [ ]:
# Record torch BEFORE installing: pyiqa drags in timm/scipy/opencv and pip is free to
# swap the CUDA torch wheel for a CPU one to satisfy them. That would not raise — it would
# just silently run the whole eval on CPU (or fail at .cuda()), so we pin and verify.
import torch as _t0
_TORCH_BEFORE, _CUDA_BEFORE = _t0.__version__, _t0.cuda.is_available()
print(f'before: torch {_TORCH_BEFORE}, cuda {_CUDA_BEFORE}')

# pyiqa provides DISTS; ptflops/einops/pytorch_msssim are UNGUARDED imports pulled in by
# `import train` and `from losses import ...`. pandas/PyYAML/matplotlib ship with the image.
!pip install -q pyiqa pytorch-msssim lpips einops ptflops tabulate

# Fail-fast: check every needed import BEFORE any model or data is touched, so a bad
# install costs seconds rather than surfacing partway into a multi-hour session.
import importlib.util as _iu
_need = ['pyiqa', 'ptflops','lpips', 'einops', 'pytorch_msssim', 'torch', 'torchvision', 'numpy', 'PIL']
_missing = [m for m in _need if _iu.find_spec(m) is None]
print('dependency check:', 'all present ✅' if not _missing else f'MISSING {_missing} ❌')
assert not _missing, f'install failed for {_missing} — re-run this cell (Internet must be ON)'

# torch must be untouched. If this trips, RESTART the kernel and reinstall the pinned
# torch version first; do not continue, the numbers would not be comparable.
import subprocess, sys
_after = subprocess.run([sys.executable, '-c',
    'import torch; print(torch.__version__, torch.cuda.is_available())'],
    capture_output=True, text=True).stdout.split()
print(f'after:  torch {_after[0]}, cuda {_after[1]}')
assert _after[0] == _TORCH_BEFORE, (
    f'pip changed torch {_TORCH_BEFORE} -> {_after[0]} — restart kernel and pin torch')
assert _after[1] == str(_CUDA_BEFORE), (
    f'CUDA availability changed {_CUDA_BEFORE} -> {_after[1]} — a CPU-only wheel was installed')
print('torch unchanged ✅')

## 2. Paths + hard guard (refuses stale physir-code)

In [ ]:
import os, sys, glob, re, json, shutil
import numpy as np, torch

INPUT_BASE = '/kaggle/input'
WORK = '/kaggle/working'
PERIMG = os.path.join(WORK, 'per_image_dists');           os.makedirs(PERIMG, exist_ok=True)
STRIP  = os.path.join(WORK, 'checkpoints_stripped'); os.makedirs(STRIP, exist_ok=True)

ROOTS = sorted(d for d in glob.glob(os.path.join(INPUT_BASE,'*')) if os.path.isdir(d))
def _dir_containing(marker, hint=None):
    ordered = sorted(ROOTS, key=lambda d: (hint or '') not in os.path.basename(d))
    for r in ordered:
        if os.path.exists(os.path.join(r, marker)): return r
        hits = sorted(glob.glob(os.path.join(r,'**',marker), recursive=True))
        if hits: return hits[0][:-len(marker)].rstrip(os.sep) or r
    return None

CODE_ROOT = _dir_containing('train.py', 'physir-code')
LOLBLUR   = _dir_containing(os.path.join('test','low_blur_noise'), 'lolblur')
assert CODE_ROOT and LOLBLUR, f'code/data not found among {ROOTS}'
if CODE_ROOT not in sys.path: sys.path.insert(0, CODE_ROOT)

# ── HARD GUARD: the uploaded loader must NOT shuffle the test set ─────────────
_ds = open(os.path.join(CODE_ROOT,'data','dataset_reader','dataset_LOLBlur.py')).read()
_bad = [d for d in re.findall(r'DataLoader\([^)]*\)', _ds, re.S)
        if 'shuffle=True' in d.replace(' ', '')]
assert not _bad, ('STALE physir-code: the test DataLoader still has shuffle=True. '
                  'Re-upload physir-code with the shuffle=False fix before running.')
print('✅ guard passed — uploaded loader does not shuffle the test set')

def find_ckpt(fname):
    # Checkpoints live in `physir-models-evaluations-set` (the s1337 pair was appended to it),
    # but the search is name-based across every attached input at any depth, so the dataset
    # layout is irrelevant — nothing breaks if files move or the slug changes.
    ordered = sorted(ROOTS, key=lambda r: 'models' not in os.path.basename(r).lower())
    for r in ordered:
        for h in sorted(glob.glob(os.path.join(r, '**', fname), recursive=True)):
            if h.endswith('.pt'): return h
    return None
print('attached:', [os.path.basename(r) for r in ROOTS])

## 2b. Session restore (attach the previous version output to resume)

In [ ]:
# ── Session restore: carry finished / partly-finished CSVs back into the working dir ──
# Continuation sessions: Add Input → Your Work → this notebook (previous version's output).
# Completed arms are then skipped; a partly-written CSV resumes mid-arm (see the eval loop).
import shutil
def _restore():
    n = 0
    for pat in ('/kaggle/input/*/per_image_dists_seeds/*.csv', '/kaggle/input/*/*/per_image_dists_seeds/*.csv'):
        for src in glob.glob(pat):
            dst = os.path.join(PERIMG, os.path.basename(src))
            if not os.path.exists(dst):
                shutil.copy2(src, dst); n += 1
    if n:
        print(f'restored {n} CSV(s) from a previous session:')
        for f in sorted(os.listdir(PERIMG)):
            rows = sum(1 for _ in open(os.path.join(PERIMG, f))) - 1
            print(f'    {f}  {rows}/1800 rows' + ('  (complete)' if rows >= 1800 else '  → will resume'))
    else:
        print('fresh session — nothing to restore')
_restore()

## 3. Targets — 8 seed replicates, multi-convention checkpoint discovery

In [ ]:
# The replicates were produced across Kaggle AND Colab and the filenames are not guaranteed
# identical, so each target is probed under several conventions. A genuinely absent checkpoint
# is REPORTED and SKIPPED — never silently dropped, which would leave a hole in the seed table
# that nothing downstream would flag.
PERIMG = os.path.join(WORK, 'per_image_dists_seeds'); os.makedirs(PERIMG, exist_ok=True)

ARMS  = ['abl_baseline', 'abl_full', 'abl_percep', 'abl_full_percep']
SEEDS = [1337, 2024]

def ckpt_candidates(arm, seed):
    """@last first — it is the operating point the seed-42 ablation table reports."""
    return [f'{arm}_s{seed}_latest.pt',        # stage6 notebooks' own convention
            f'{arm}_s{seed}_last.pt',
            f'{arm}_s{seed}.pt',
            f'{arm}_{seed}_latest.pt']

TARGETS, MISSING = [], []
print(f'{"target":26s} {"status":10s} checkpoint')
print('-' * 78)
for seed in SEEDS:
    for arm in ARMS:
        name = f'{arm}_s{seed}'
        out  = os.path.join(PERIMG, f'per_image_dists_{name}.csv')
        hit  = next((p for p in (find_ckpt(c) for c in ckpt_candidates(arm, seed)) if p), None)
        if os.path.exists(out):
            st = 'DONE ✅'
        elif hit:
            st = 'found'
        else:
            st = 'MISSING ❌'; MISSING.append(name)
        print(f'{name:26s} {st:10s} {os.path.relpath(hit, INPUT_BASE) if hit else "—"}')
        if hit or os.path.exists(out):
            TARGETS.append((name, os.path.basename(hit) if hit else None))
print('-' * 78)
print(f'{len(TARGETS)}/8 evaluable · {len(MISSING)} missing')
if MISSING:
    print('\n⚠ Missing checkpoints — attach the Stage-6 notebook outputs that hold them:')
    for m in MISSING: print('   ', m)
    print('  (Add Input → Your Work → stage6_seed_replicates_kaggle / '
          'stage6_seed1337_completion_kaggle / stage6_replicates_colab)')
    print('  The run proceeds with whatever IS present; the coverage manifest records the rest,')
    print('  so Ch.4 states DISTS scope from data instead of memory.')

## 4. Eval engine — lifted unchanged from `dists_eval_kaggle.ipynb`

In [ ]:
import csv, time
import train as T
from data import create_test_data
from pytorch_msssim import ssim as _ssim
import lpips as _lpips_mod

BATCH = 4          # uniform 1120x640 test images → batching is safe and ~3x faster
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# The DarkIR-convention LPIPS column exists only for the cross-paper comparison; the seed
# replicates never use it. Skipping it on CPU halves the dominant cost (two VGG passes -> one).
DUAL_LPIPS = (dev == 'cuda')
print('device:', dev, '| batch', BATCH,
      '| dual-LPIPS', 'on' if DUAL_LPIPS else 'OFF (CPU: not needed for replicate arms)')
if dev != 'cuda':
    torch.set_num_threads(os.cpu_count() or 4)
    print(f'  CPU run: {os.cpu_count()} threads. Expect ~3-6 h per arm; the loop is idempotent,')
    print('  so a cut session resumes from the next unfinished CSV.')

opt_test = {'name':'LOLBlur',
            'val':{'test_path': os.path.join(LOLBLUR,'test'), 'batch_size_test': BATCH},
            'train':{'n_workers':2, 'verbose':False}}
raw,_ = create_test_data(rank=0, world_size=1, opt=opt_test)
LOADER = raw if not isinstance(raw, dict) else list(raw.values())[0]['loader']
DS = LOADER.dataset
# clip-qualified so the key is UNIQUE: LOLBlur test basenames repeat across its 30 clips
FNAMES = [os.path.join(os.path.basename(os.path.dirname(p)), os.path.basename(p))
          for p in DS.imgs_low]                          # dataset sorts its lists
assert len(set(FNAMES)) == len(FNAMES), 'image keys are not unique'
print(f'test set: {len(DS)} images | first: {FNAMES[0]} | last: {FNAMES[-1]}')

# runtime proof that iteration order == sorted order (i.e. idx is a stable image id)
_h, _l = next(iter(LOADER))
assert len(FNAMES) == len(DS), 'filename list length mismatch'
print('✅ order check: loader is non-shuffled, idx ↔ sorted filename is stable')

# (LPIPS is not used here — this notebook computes DISTS only.)
NET = {'name':'DarkIR','img_channels':3,'width':32,'middle_blk_num_enc':2,'middle_blk_num_dec':2,
       'enc_blk_nums':[1,2,3],'dec_blk_nums':[3,1,1],'dilations':[1,4,9],'extra_depth_wise':True}

def load_model(path):
    ck = torch.load(path, map_location='cpu', weights_only=False)
    st = (ck.get('ema') or ck.get('model') or ck.get('params')) if isinstance(ck, dict) else ck
    assert st is not None, f'no weights in {path}'
    m = T.build_model(NET).to(dev)
    m.load_state_dict({k.replace('module.',''):v for k,v in st.items()}, strict=False)
    return m.eval(), ck

# ── DISTS setup + the convention self-check ─────────────────────────────────
import pyiqa, json as _json
DISTS = pyiqa.create_metric('dists', device=dev, as_loss=False)
print('pyiqa', pyiqa.__version__, '| dists lower_better =', DISTS.lower_better)

# ── Guards that actually test something ────────────────────────────────────
# NOTE: "DISTS(x,x) ~ 0" is NOT a convention check — identical inputs score 0 at ANY input
# range, so that assert would pass even if we wrongly fed [-1,1] tensors. The real protections
# are (a) pyiqa's own check_input_range, verified live below, and (b) the fact that `pred` is
# clamp(0,1) and `high` comes from ToTensor, both already in [0,1].
_x = torch.rand(2, 3, 128, 128, device=dev)
_y = torch.rand_like(_x)

# (i) metric is alive and discriminating, and returns ONE SCORE PER IMAGE (not a batch mean)
_same, _diff = DISTS(_x, _x).flatten(), DISTS(_x, _y).flatten()
assert _same.numel() == 2 and _diff.numel() == 2, \
    f'DISTS returned {_diff.numel()} score(s) for a batch of 2 — per-sample output required'
assert float(_same.mean()) < 1e-3, 'DISTS(x,x) is not ~0 — metric is misconfigured'
assert float(_diff.mean()) > float(_same.mean()), 'DISTS is not discriminating'

# (ii) the range guard is ACTIVE: feeding LPIPS-style [-1,1] must raise, not silently score.
_range_guard = False
try:
    DISTS(_x * 2 - 1, _y * 2 - 1)
except AssertionError:
    _range_guard = True
assert _range_guard, ('pyiqa check_input_range is OFF — a wrong input range would score '
                      'silently. Re-create the metric with check_input_range=True.')
print(f'self-check: per-sample ✅ | DISTS(x,x)={float(_same.mean()):.6f} | '
      f'DISTS(x,y)={float(_diff.mean()):.4f} | [0,1] range guard ACTIVE ✅')
PROV = {'metric': 'DISTS', 'impl': f'pyiqa {pyiqa.__version__} (create_metric("dists"))',
        'backbone': 'VGG16 (shared with LPIPS-vgg — not an independent feature space)',
        'input_range': '[0,1] (ImageNet normalisation applied internally by pyiqa)',
        'lower_better': bool(DISTS.lower_better),
        'check_input_range': True,
        'resizing': 'none — scored at native 1120x640',
        'per_sample_output': True,
        'self_check_identical': float(_same.mean()),
        'self_check_random': float(_diff.mean())}

HEADER_COLS = ['idx', 'fname', 'dists']

def _rows_done(path):
    """Completed rows, truncated to a whole-batch boundary so a half-written batch is redone."""
    if not os.path.exists(path): return 0
    rows = list(csv.reader(open(path)))[1:]
    done = (len(rows) // BATCH) * BATCH
    if done != len(rows):
        with open(path, 'w', newline='') as f:
            w = csv.writer(f); w.writerow(HEADER_COLS); w.writerows(rows[:done])
    return done

@torch.no_grad()
def eval_dists(model, out_csv):
    done = _rows_done(out_csv)
    if done: print(f'    resuming at {done}/{len(DS)}')
    fh = open(out_csv, 'a' if done else 'w', newline=''); w = csv.writer(fh)
    if not done: w.writerow(HEADER_COLS); fh.flush()
    i0, t0 = 0, time.time()
    for high, low in LOADER:
        n = high.shape[0]
        if i0 + n <= done:
            i0 += n; continue
        high, low = high.to(dev, non_blocking=True), low.to(dev, non_blocking=True)
        pred = model(low, side_loss=False).clamp(0, 1)[:, :, :high.shape[2], :high.shape[3]]
        if i0 == done:   # once per model: prove the real tensors are in the expected range
            assert 0.0 <= float(high.min()) and float(high.max()) <= 1.0, 'GT not in [0,1]'
            assert 0.0 <= float(pred.min()) and float(pred.max()) <= 1.0, 'prediction not in [0,1]'
        d = DISTS(pred, high).flatten()          # [0,1] inputs — see the convention note above
        for k in range(n):
            w.writerow([i0 + k, FNAMES[i0 + k], float(d[k])])
        i0 += n
        if i0 % 200 < BATCH:
            fh.flush(); os.fsync(fh.fileno())
            r = (time.time() - t0) / max(i0 - done, 1)
            print(f'    {i0}/{len(DS)}  ~{r * (len(DS) - i0) / 60:.1f} min left', flush=True)
    fh.close()
    vals = [float(r[2]) for r in list(csv.reader(open(out_csv)))[1:]]
    return sum(vals) / len(vals), len(vals)


## 5. RUN — idempotent, resumable

In [ ]:
for name, fn in TARGETS:
    out = os.path.join(PERIMG, f'per_image_dists_{name}.csv')
    if _rows_done(out) >= len(DS):
        print(f'✅ {name}: complete — skip'); continue
    p = find_ckpt(fn) if fn else None
    if not p:
        print(f'⏭  {name}: checkpoint not attached — skip'); continue
    print(f'\n=== DISTS {name}  ({os.path.relpath(p, INPUT_BASE)}) ===', flush=True)
    model, _ = load_model(p)
    mean_d, n = eval_dists(model, out)
    print(f'  → DISTS {mean_d:.4f}  (n={n}, lower is better)')
    del model; torch.cuda.empty_cache()

print('\nDISTS by arm x seed (lower = better; seed 42 comes from dists_eval_kaggle):')
print(f'{"arm":20s} {"s1337":>9s} {"s2024":>9s}')
for arm in ARMS:
    row = f'{arm:20s}'
    for seed in SEEDS:
        f = os.path.join(PERIMG, f'per_image_dists_{arm}_s{seed}.csv')
        if _rows_done(f) >= len(DS):
            v = [float(r[2]) for r in list(csv.reader(open(f)))[1:]]
            row += f'  {sum(v)/len(v):9.4f}'
        else:
            row += f'  {"—":>9s}'
    print(row)

## 6. Coverage manifest + bundle

In [ ]:
# Bundling must never raise: two earlier sessions ended with a check failing AFTER the eval
# had succeeded. Diagnostics are wrapped; only the archive is required.
done = sorted(f for f in os.listdir(PERIMG)
              if f.endswith('.csv') and _rows_done(os.path.join(PERIMG, f)) >= len(DS))
part = sorted(set(f for f in os.listdir(PERIMG) if f.endswith('.csv')) - set(done))

try:
    print('fname-order check across completed arms (all must share one order to join on idx):')
    ref = None
    for f in done:
        names = [r['fname'] for r in csv.DictReader(open(os.path.join(PERIMG, f)))]
        if ref is None: ref = names; print(f'   reference: {f}')
        elif names != ref: print(f'   ❌ {f}: DIFFERENT ORDER — do not join')
    print(f'   ✅ {len(done)} file(s) share one fname order' if ref else '   (nothing complete yet)')
except Exception as e:
    print(f'⚠ order check skipped ({type(e).__name__}: {e})')

COVERAGE = {'scope': 'DISTS on Stage-6 seed replicates @last, LOLBlur test',
            'arms': ARMS, 'seeds_here': SEEDS, 'seed_42_source': 'dists_eval_kaggle.ipynb',
            'complete': [f.replace('per_image_dists_', '').replace('.csv', '') for f in done],
            'partial':  [f.replace('per_image_dists_', '').replace('.csv', '') for f in part],
            'missing_checkpoints': MISSING,
            'not_covered': {'384_anchors': 'single-seed by construction',
                            'generalisation_sets': 'seed 42 only (dists_eval_kaggle section 6)',
                            'Real-LOLBlur': 'excluded — DISTS is full-reference, no GT'}}
_json.dump(COVERAGE, open(os.path.join(WORK, 'dists_seeds_coverage.json'), 'w'), indent=2)
print(f'\n{len(done)} complete, {len(part)} partial, {len(MISSING)} missing checkpoint(s)')
for f in done: print('   ✅', f)
for f in part: print('   … ', f, '(resumes next session)')

if os.listdir(PERIMG):
    try:
        z = shutil.make_archive(os.path.join(WORK, 'per_image_dists_seeds'), 'zip',
                                WORK, os.path.basename(PERIMG))
        print(f'\n✅ archived → {z}  ({os.path.getsize(z)/2**20:.2f} MB)')
    except Exception as e:
        print(f'\n⚠ make_archive failed ({type(e).__name__}: {e}) — writing zip directly')
        import zipfile
        z = os.path.join(WORK, 'per_image_dists_seeds.zip')
        with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as zf:
            for f in os.listdir(PERIMG):
                zf.write(os.path.join(PERIMG, f), os.path.join('per_image_dists_seeds', f))
        print(f'✅ archived → {z}')
else:
    print('\n❌ nothing produced — see the target table above for missing checkpoints.')
print('coverage manifest → dists_seeds_coverage.json')